This notebook focuses on key Delta Lake maintence oprations

**Optimizing Tables:** Use the OPTIMIZE command to compact small files and improve query performance.

**ZORDER Optimization:** Apply ZORDER BY to colocate related data, enabling efficient data skipping and faster queries.

**Running VACUUM:** Remove obsolete files and reclaim storage space by executing the VACUUM command.

These features help maintain efficient, performant Delta tables in your data lake.

VACUUM table_name   -- vacuum files not required by versions older than the default retention period

VACUUM table_name DRY RUN    -- do dry run to get the list of files to be deleted

In [0]:
%sql
drop table if exists training.delta_demo.people_delta_demo

In [0]:
#create initial data with sample
data = [(1,"Alice", 10), (2,"Bob", 20), (3,"Charlie", 30)]
df = spark.createDataFrame(data, ["id","name", "age"])
#write data to delta
df.write.format("delta").mode("overwrite").saveAsTable("training.delta_demo.people_delta_demo")

In [0]:
delta_df = spark.read.table("training.delta_demo.people_delta_demo")
display(delta_df)

In [0]:
delta_df.write.format("delta").mode("append").save("/Volumes/training/delta_demo/delta_demo/mytable")

In [0]:
df2= spark.read.format("delta").load("/Volumes/training/delta_demo/delta_demo/mytable")
display(df2)

In [0]:
%sql
optimize '/Volumes/training/delta_demo/delta_demo/mytable'

In [0]:
%sql
optimize training.delta_demo.people_delta_demo

In [0]:
%sql
optimize '/Volumes/training/delta_demo/delta_demo/mytable' zorder by id

In [0]:
%sql
optimize training.delta_demo.people_delta_demo zorder by id

In [0]:
%sql
describe detail training.delta_demo.people_delta_demo

In [0]:
%sql
DESCRIBE HISTORY training.delta_demo.people_delta_demo;
    
DESCRIBE DETAIL training.delta_demo.people_delta_demo;
    
DESCRIBE EXTENDED training.delta_demo.people_delta_demo;
    


In [0]:
%sql
VACUUM training.delta_demo.people_delta_demo DRY RUN

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In Delta Lake (used in Databricks, Spark, etc.), the VACUUM retention configuration controls how long old, unreferenced data files are kept before being physically deleted.

1. Default Behavior
Default retention: 7 days (168 hours).
Purpose: Allows time travel and rollback to older table versions.

In [0]:
#Global Spark Config
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.conf.set("spark.databricks.delta.deletedFileRetentionDuration", "interval 1 day")


In [0]:
g